# Notebook 4 — Modèle BK N blocs : séries temporelles et portraits de phase

Reproduction des Figures 3, 4, 5 d'Erickson et al. (2011).

**Système :** N blocs couplés avec friction RSF (slip law), formulation non-dimensionnelle (éq. 9).  
**Paramètres :** γ_μ = 0.5, γ_λ = √0.2, ξ = 0.5, f̃ = 3.2, ε = 0.5  
**Positions (cell-centered) :** x̄_j = (j − 0.5) × 20/N, j = 1…N  
**Intégrateur :** RK45 Dormand–Prince maison (adaptatif, ~17× plus rapide que Radau)

| Figure | N  | Régime          | Durée |
|--------|----|-----------------|-------|
| Fig. 3 | 3  | Périodique      | 300   |
| Fig. 4 | 9  | Quasi-périodique| 1500  |
| Fig. 5 | 20 | Chaotique       | 1500  |

Chaque figure contient : position centrale, vitesse centrale, variable d'état θ̄, 
espace de phases 3D (ū, v̄, Θ̄) du bloc central.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import gc

# ══════════════════════════════════════════════════════════════════════════════
#  Dormand–Prince RK45 coefficients
# ══════════════════════════════════════════════════════════════════════════════
A_tab = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [1/5, 0, 0, 0, 0, 0, 0],
    [3/40, 9/40, 0, 0, 0, 0, 0],
    [44/45, -56/15, 32/9, 0, 0, 0, 0],
    [19372/6561, -25360/2187, 64448/6561, -212/729, 0, 0, 0],
    [9017/3168, -355/33, 46732/5247, 49/176, -5103/18656, 0, 0],
    [35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0],
])
B5 = np.array([35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0])
B4 = np.array([5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40])
C  = np.array([0, 1/5, 3/10, 4/5, 8/9, 1, 1])


def rk45_step(f, t, y, h):
    n_stages = 7
    k = np.zeros((n_stages, len(y)))
    k[0] = f(t, y)
    for i in range(1, n_stages):
        yi = y + h * np.dot(A_tab[i, :i], k[:i])
        k[i] = f(t + C[i] * h, yi)
    y5 = y + h * np.dot(B5, k)
    y4 = y + h * np.dot(B4, k)
    err = np.abs(y5 - y4)
    return y5, err, k[0]


def rk45_integrate(f, t_span, y0, rtol=1e-8, atol=1e-10,
                   h_init=1e-3, h_min=1e-14, h_max=10.0,
                   max_steps=100_000_000, store_every=1):
    t0, tf = t_span
    y = np.array(y0, dtype=float)
    t = t0
    h = min(h_init, tf - t0)

    chunk = 200_000
    t_list = np.zeros(chunk)
    y_list = np.zeros((chunk, len(y)))
    t_list[0] = t
    y_list[0] = y
    idx = 1
    n_alloc = chunk
    step_count = 0

    safety = 0.9
    n_steps = 0
    n_reject = 0

    while t < tf:
        if n_steps >= max_steps:
            print(f"  WARNING: hit max_steps={max_steps:,}")
            break

        h = min(h, tf - t)
        if h < h_min:
            h = h_min

        y_new, err, _ = rk45_step(f, t, y, h)

        scale = atol + rtol * np.maximum(np.abs(y), np.abs(y_new))
        err_norm = np.sqrt(np.mean((err / scale)**2))

        if err_norm <= 1.0:
            t = t + h
            y = y_new
            n_steps += 1
            step_count += 1

            if step_count % store_every == 0:
                if idx >= n_alloc:
                    t_list = np.concatenate([t_list, np.zeros(chunk)])
                    y_list = np.concatenate([y_list, np.zeros((chunk, len(y)))])
                    n_alloc += chunk
                t_list[idx] = t
                y_list[idx] = y
                idx += 1

            if err_norm < 1e-30:
                h = h * 5
            else:
                h = h * safety * err_norm**(-0.2)
            h = min(h, h_max)
        else:
            h = h * safety * err_norm**(-0.25)
            h = max(h, h_min)
            n_reject += 1

    t_list = t_list[:idx]
    y_list = y_list[:idx]
    print(f"  {n_steps:,} steps ({n_reject:,} rejected), {idx:,} stored")
    return t_list, y_list


# ══════════════════════════════════════════════════════════════════════════════
#  N-block BK model — non-dimensional eq. (9) from Erickson et al. (2011)
# ══════════════════════════════════════════════════════════════════════════════

def make_nblock_rhs(N, gamma_mu, gamma_lam, xi, eps, f_bar):
    gm2 = gamma_mu**2
    gl2 = gamma_lam**2
    coeff_fric = gm2 / xi
    eps1 = 1.0 + eps

    def rhs(t, y):
        u = y[:N]
        v = y[N:2*N]
        Theta = y[2*N:3*N]

        dy = np.empty(3*N)

        # du/dt = v
        dy[:N] = v

        # Laplacian with free BCs: u_0 = u_1, u_{N+1} = u_N
        lap = np.empty(N)
        lap[0]    = u[1] - u[0]
        lap[1:-1] = u[:-2] - 2*u[1:-1] + u[2:]
        lap[-1]   = u[-2] - u[-1]

        # Protect log argument
        vp1 = np.maximum(v + 1.0, 1e-30)
        ln_vp1 = np.log(vp1)

        # dv/dt
        dy[N:2*N] = gm2 * lap - gl2 * u - coeff_fric * (f_bar + Theta + ln_vp1)

        # dΘ/dt
        dy[2*N:3*N] = -vp1 * (Theta + eps1 * ln_vp1)

        return dy

    return rhs


def simulate_nblock(N, gamma_mu, gamma_lam, xi, eps, f_bar, duration,
                    rtol=1e-8, atol=1e-10, h_max=1.0, store_every=1):
    rhs = make_nblock_rhs(N, gamma_mu, gamma_lam, xi, eps, f_bar)

    # Equilibrium
    u_o = -f_bar * gamma_mu**2 / (xi * gamma_lam**2)
    print(f"  N={N}, u_o = {u_o:.4f}")

    # Blockk positions: cell-centered on chain of length 20
    # x_j = (j - 0.5) * 20/N ensures a block sits at x=10 for odd N
    x_bar = (np.arange(1, N+1) - 0.5) * (20.0 / N)

    # Gaussian IC centered at x=10
    sigma_ic = 1.0
    u_init = u_o + np.exp(-(x_bar - 10.0)**2 / sigma_ic**2)
    v_init = np.zeros(N)
    Theta_init = np.zeros(N)

    y0 = np.concatenate([u_init, v_init, Theta_init])
    print(f"  IC: u range [{u_init.min():.4f}, {u_init.max():.4f}]")

    t, y = rk45_integrate(rhs, [0, duration], y0,
                          rtol=rtol, atol=atol,
                          h_init=1e-3, h_max=h_max,
                          store_every=store_every)

    u_arr     = y[:, :N]
    v_arr     = y[:, N:2*N]
    Theta_arr = y[:, 2*N:3*N]

    return t, u_arr, v_arr, Theta_arr, x_bar, u_init


# ══════════════════════════════════════════════════════════════════════════════
#  Parameters from Erickson Figs 3–5
# ══════════════════════════════════════════════════════════════════════════════
gamma_mu  = 0.5
gamma_lam = np.sqrt(0.2)
xi        = 0.5
eps       = 0.5
f_bar     = 3.2


def plot_erickson_figure(N, duration, filename, rtol=1e-8, atol=1e-10,
                         h_max=1.0, store_every=1):
    print(f"\n{'='*60}")
    print(f"  Simulating N={N} blocks, duration={duration}")
    print(f"{'='*60}")

    t, u, v, Theta, x_bar, u_init = simulate_nblock(
        N, gamma_mu, gamma_lam, xi, eps, f_bar, duration,
        rtol=rtol, atol=atol, h_max=h_max, store_every=store_every)

    # Center block index
    mid = (N - 1) // 2

    fig = plt.figure(figsize=(14, 12))

    # ── (a) Initial data ────────────────────────────────────────────
    ax_a = fig.add_subplot(2, 2, 1)
    ax_a.plot(x_bar, u_init, 'bo', markersize=6)
    ax_a.set_xlabel(r'$\bar{x}_j$')
    ax_a.set_ylabel(r'$\bar{u}_j(0)$')
    ax_a.set_title(f'Initial Data for {N} Blockk System')
    ax_a.set_xlim(0, 20)
    ax_a.grid(True, alpha=0.3)

    # ── (b) 3D waterfall of all blocks' slip vs time ────────────────
    ax_b = fig.add_subplot(2, 2, 2, projection='3d')

    max_pts = 3000
    if len(t) > max_pts:
        idx_dec = np.linspace(0, len(t)-1, max_pts, dtype=int)
    else:
        idx_dec = np.arange(len(t))

    t_dec = t[idx_dec]
    u_dec = u[idx_dec, :]

    for j in range(N):
        ax_b.plot(t_dec, np.full_like(t_dec, x_bar[j]), u_dec[:, j],
                  lw=0.4, alpha=0.6)

    ax_b.set_xlabel(r'$\bar{t}$', labelpad=10)
    ax_b.set_ylabel(r'$\bar{x}_j$', labelpad=10)
    ax_b.set_zlabel(r'$\bar{u}_j$', labelpad=5)
    ax_b.set_title(f'Slip for {N} Blockk System')
    ax_b.view_init(elev=25, azim=-50)

    # ── (c) Center block slip vs time (after transient) ──────────────
    ax_c = fig.add_subplot(2, 2, 3)

    # Keep roughly last 70% to see settled behavior
    t_trans = 0.3 * duration
    mask = t > t_trans
    ax_c.plot(t[mask], u[mask, mid], 'b-', lw=0.8)
    ax_c.set_xlabel(r'$\bar{t}$')
    ax_c.set_ylabel(r'$\bar{u}_{%d}(\bar{t})$' % (mid+1))
    ax_c.set_title(f'Contour for Center Blockk of {N} Blockk System')
    ax_c.grid(True, alpha=0.3)

    # ── (d) Phase space of center block ─────────────────────────────
    ax_d = fig.add_subplot(2, 2, 4, projection='3d')

    u_mid = u[mask, mid]
    v_mid = v[mask, mid]
    T_mid = Theta[mask, mid]

    max_phase_pts = 8000
    if len(u_mid) > max_phase_pts:
        idx_ph = np.linspace(0, len(u_mid)-1, max_phase_pts, dtype=int)
    else:
        idx_ph = np.arange(len(u_mid))

    # Phase space: axes order (v, u, Θ) to match Erickson layout
    ax_d.plot(v_mid[idx_ph], u_mid[idx_ph], T_mid[idx_ph], lw=0.3, alpha=0.6)
    ax_d.set_xlabel(r'$\bar{v}_{%d}(\bar{t})$' % (mid+1), labelpad=8)
    ax_d.set_ylabel(r'$\bar{u}_{%d}(\bar{t})$' % (mid+1), labelpad=8)
    ax_d.set_zlabel(r'$\bar{\Theta}_{%d}(\bar{t})$' % (mid+1), labelpad=5)
    ax_d.set_title(f'Phase Space for Center Blockk of {N} Blockk System')
    ax_d.view_init(elev=25, azim=-60)

    fig.suptitle(
        rf'Erickson et al. (2011) — $N={N}$, $\varepsilon={eps}$, '
        rf'$\xi={xi}$, $\gamma_{{\mu}}={gamma_mu}$, '
        rf'$\gamma_{{\lambda}}=\sqrt{{0.2}}$, $\bar{{f}}={f_bar}$',
        fontsize=12, fontweight='bold', y=1.02)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {filename}")

    del t, u, v, Theta
    gc.collect()


# ══════════════════════════════════════════════════════════════════════════════
#  Run: N=3 (Fig 3), N=9 (Fig 4), N=20 (Fig 5)
# ══════════════════════════════════════════════════════════════════════════════

# Fig 3: N=3, periodic — duration ~300
plot_erickson_figure(N=3, duration=300, filename='fig3_N3.png',
                     rtol=1e-8, atol=1e-10, h_max=0.5)

# Fig 4: N=9, periodic with period doubling — duration ~1500
plot_erickson_figure(N=9, duration=1500, filename='fig4_N9.png',
                     rtol=1e-8, atol=1e-10, h_max=0.5)

# Fig 5: N=20, chaotic — duration ~1500
plot_erickson_figure(N=20, duration=1500, filename='fig5_N20.png',
                     rtol=1e-8, atol=1e-10, h_max=0.5, store_every=1)

print("\nAll done!")


  Simulating N=3 blocks, duration=300
  N=3, u_o = -8.0000
  IC: u range [-8.0000, -7.0000]
  2,485 steps (136 rejected), 2,486 stored
  Saved: fig3_N3.png

  Simulating N=9 blocks, duration=1500
  N=9, u_o = -8.0000
  IC: u range [-8.0000, -7.0000]
  14,343 steps (917 rejected), 14,344 stored
  Saved: fig4_N9.png

  Simulating N=20 blocks, duration=1500
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  16,287 steps (1,172 rejected), 16,288 stored
  Saved: fig5_N20.png

All done!
